In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta

def fetch_mlb_scores(target_date):
    # Format the date as YYYY-MM-DD
    date_str = target_date.strftime('%Y-%m-%d')
    url = f'https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={date_str}'

    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to fetch data: {response.status_code}")
        return None

    data = response.json()
    games = data.get('dates', [])[0].get('games', []) if data.get('dates') else []

    game_data = []
    for game in games:
        game_info = {
            'Date': date_str,
            'Home_Team': game['teams']['home']['team']['name'],
            'Home_Score': game['teams']['home']['score'],
            'Away_Team': game['teams']['away']['team']['name'],
            'Away_Score': game['teams']['away']['score'],
            'Status': game['status']['detailedState']
        }
        game_data.append(game_info)

    return pd.DataFrame(game_data)

# Example usage:
# Fetch scores for yesterday
yesterday = datetime.now() - timedelta(days=1)
scores_df = fetch_mlb_scores(yesterday)

if scores_df is not None:
    print(scores_df)
    # Save to CSV
    scores_df.to_csv(f'mlb_scores_{yesterday.strftime("%Y-%m-%d")}.csv', index=False)


          Date             Home_Team  Home_Score             Away_Team  \
0   2025-05-26        Detroit Tigers           3  San Francisco Giants   
1   2025-05-26     Milwaukee Brewers           3        Boston Red Sox   
2   2025-05-26          Chicago Cubs           3      Colorado Rockies   
3   2025-05-26     Baltimore Orioles           5   St. Louis Cardinals   
4   2025-05-26         Texas Rangers           1     Toronto Blue Jays   
5   2025-05-26    Kansas City Royals           4       Cincinnati Reds   
6   2025-05-26         New York Mets           2     Chicago White Sox   
7   2025-05-26   Cleveland Guardians           1   Los Angeles Dodgers   
8   2025-05-26        Tampa Bay Rays           0       Minnesota Twins   
9   2025-05-26  Arizona Diamondbacks           0    Pittsburgh Pirates   
10  2025-05-26      San Diego Padres           0         Miami Marlins   
11  2025-05-26    Los Angeles Angels           0      New York Yankees   

    Away_Score       Status  
0      

In [2]:
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import date, timedelta

# Determine yesterday's date
yesterday = date.today() - timedelta(days=1)
yesterday_str = yesterday.isoformat()

# Construct the URL for the desired date
url = f"https://www.baseball-reference.com/boxes/?date={yesterday_str}"

# Fetch the page content
response = requests.get(url)
if response.status_code != 200:
    print(f"Failed to retrieve data: Status code {response.status_code}")
    exit()

# Parse the HTML content
soup = BeautifulSoup(response.text, 'html.parser')

# Find all game summaries
games = soup.find_all('div', class_='game_summary')

# Extract game data
game_data = []
for game in games:
    teams = game.find_all('tr')
    if len(teams) >= 3:
        away_team = teams[1].find('td').text.strip()
        away_score = teams[1].find_all('td')[1].text.strip()
        home_team = teams[2].find('td').text.strip()
        home_score = teams[2].find_all('td')[1].text.strip()
        
        game_data.append({
            'Date': yesterday_str,
            'Away Team': away_team,
            'Away Score': away_score,
            'Home Team': home_team,
            'Home Score': home_score
        })

# Create a DataFrame
df = pd.DataFrame(game_data)

# Ensure the 'data' directory exists
output_dir = os.path.join(os.getcwd(), 'data')
os.makedirs(output_dir, exist_ok=True)

# Define the output file path
output_file = os.path.join(output_dir, f'game_scores_{yesterday_str}.csv')

# Save the DataFrame to a CSV file
df.to_csv(output_file, index=False)
print(f"Game scores for {yesterday_str} have been saved to {output_file}")


Game scores for 2025-05-26 have been saved to /workspaces/MLB_Model/mlb-api/data/game_scores_2025-05-26.csv


In [3]:
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import date, timedelta

# ==== CONFIG ====
target_date = date.today() - timedelta(days=1)  # change to date(2025, 5, 26) for manual testing
target_str = target_date.isoformat()
output_folder = "data"
output_file = os.path.join(output_folder, f"game_scores_{target_str}.csv")
os.makedirs(output_folder, exist_ok=True)

# ==== STEP 1: GET INDEX PAGE WITH GAME LINKS ====
index_url = f"https://www.baseball-reference.com/boxes/?date={target_str}"
response = requests.get(index_url)
soup = BeautifulSoup(response.text, 'html.parser')

# ==== STEP 2: Extract box score links ====
boxscore_links = []
for link in soup.find_all("a", href=True):
    if "/boxes/" in link["href"] and link.text.strip() == "Boxscore":
        boxscore_links.append("https://www.baseball-reference.com" + link["href"])

print(f"🔗 Found {len(boxscore_links)} boxscores")

# ==== STEP 3: Visit each boxscore page and extract scores ====
game_rows = []

for url in boxscore_links:
    box_resp = requests.get(url)
    box_soup = BeautifulSoup(box_resp.text, 'html.parser')

    # Get the linescore table
    linescore_table = box_soup.find('table', {'class': 'linescore'})

    if not linescore_table:
        print(f"⚠️ Skipping {url} — no linescore found")
        continue

    rows = linescore_table.find_all('tr')
    if len(rows) < 3:
        continue

    try:
        away_row = rows[0]
        home_row = rows[1]

        away_team = away_row.find('th').text.strip()
        home_team = home_row.find('th').text.strip()

        away_score = int(away_row.find_all('td')[-1].text.strip())
        home_score = int(home_row.find_all('td')[-1].text.strip())

        game_rows.append({
            "Date": target_str,
            "Away Team": away_team,
            "Away Score": away_score,
            "Home Team": home_team,
            "Home Score": home_score
        })

    except Exception as e:
        print(f"❌ Error parsing game: {url} - {e}")
        continue

# ==== STEP 4: Save to CSV ====
df = pd.DataFrame(game_rows)
df.to_csv(output_file, index=False)
print(f"✅ Saved {len(df)} games to {output_file}")
print(df.head())


🔗 Found 0 boxscores
✅ Saved 0 games to data/game_scores_2025-05-26.csv
Empty DataFrame
Columns: []
Index: []


In [4]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta

def fetch_mlb_scores(target_date):
    date_str = target_date.strftime('%Y-%m-%d')
    url = f'https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={date_str}'

    response = requests.get(url)
    if response.status_code != 200:
        print(f"❌ Failed to fetch data: {response.status_code}")
        return None

    data = response.json()
    games = data.get('dates', [])[0].get('games', []) if data.get('dates') else []

    game_data = []
    for game in games:
        game_info = {
            'Date': date_str,
            'Home Team': game['teams']['home']['team']['name'],
            'Home Score': game['teams']['home']['score'],
            'Away Team': game['teams']['away']['team']['name'],
            'Away Score': game['teams']['away']['score'],
            'Status': game['status']['detailedState']
        }
        game_data.append(game_info)

    return pd.DataFrame(game_data)

# ==== MAIN ====
# Fetch scores for yesterday
yesterday = datetime.now() - timedelta(days=1)
scores_df = fetch_mlb_scores(yesterday)

if scores_df is not None:
    print(scores_df)

    # Ensure output dir exists
    output_dir = os.path.join("data")
    os.makedirs(output_dir, exist_ok=True)

    # Save to file
    output_file = os.path.join(output_dir, f'game_scores_{yesterday.strftime("%Y-%m-%d")}.csv')
    scores_df.to_csv(output_file, index=False)
    print(f"✅ Saved scores to {output_file}")


          Date             Home Team  Home Score             Away Team  \
0   2025-05-26        Detroit Tigers           3  San Francisco Giants   
1   2025-05-26     Milwaukee Brewers           3        Boston Red Sox   
2   2025-05-26          Chicago Cubs           3      Colorado Rockies   
3   2025-05-26     Baltimore Orioles           5   St. Louis Cardinals   
4   2025-05-26         Texas Rangers           1     Toronto Blue Jays   
5   2025-05-26    Kansas City Royals           4       Cincinnati Reds   
6   2025-05-26         New York Mets           2     Chicago White Sox   
7   2025-05-26   Cleveland Guardians           2   Los Angeles Dodgers   
8   2025-05-26        Tampa Bay Rays           0       Minnesota Twins   
9   2025-05-26  Arizona Diamondbacks           0    Pittsburgh Pirates   
10  2025-05-26      San Diego Padres           0         Miami Marlins   
11  2025-05-26    Los Angeles Angels           0      New York Yankees   

    Away Score       Status  
0      